# Phase 1 — Data Pipeline

**Objective:** Download, clean, align, and export all raw data into standardised monthly `.parquet` files.

**Dependencies:** None (entry point)

**Outputs:**
- `factor_returns.parquet` — FF5 + Momentum (decimal returns, month-end)
- `macro_indicators.parquet` — 10 transformed macro features (month-end)
- `sp500_daily_prices.parquet` — S&P 500 adjusted close prices
- `sp500_monthly_returns.parquet` — S&P 500 monthly returns
- `master_data.parquet` — Tech portfolio OHLCV (20 tickers + benchmarks)

In [1]:
# === Setup & Imports ===
import os, sys, warnings, hashlib, json, time, logging, urllib.request
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import yfinance as yf
from dotenv import load_dotenv

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

# Set User-Agent for all urllib requests (Wikipedia, FRED CSV, etc.)
opener = urllib.request.build_opener()
opener.addheaders = [('User-Agent', 'Mozilla/5.0 (compatible; FactorTimingEngine/1.0)')]
urllib.request.install_opener(opener)

# Project paths
sys.path.insert(0, str(Path.cwd().parent))
from src.config import (
    PROJECT_ROOT, RAW_DIR, INTERIM_DIR, PROCESSED_DIR, LOGS_DIR,
    FIGURES_DIR, TABLES_DIR, TICKERS, BENCHMARKS, SHORT_HISTORY_TICKERS,
    FRED_SERIES, RANDOM_STATE, FACTOR_START, FACTOR_END, TECH_START, TECH_END
)
from src.validation import validate_parquet, check_nan_propagation, quick_data_check
from src.data_loader import (
    fetch_french_factors, fetch_sp500_tickers,
    fetch_prices_batch, fetch_tech_portfolio
)

# Logging
PHASE_NUM = 1
logging.basicConfig(
    filename=str(LOGS_DIR / f'phase_{PHASE_NUM}_{datetime.now():%Y%m%d_%H%M}.log'),
    level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s'
)
logger = logging.getLogger(__name__)
logger.info(f"Phase {PHASE_NUM} started")

load_dotenv(PROJECT_ROOT / '.env')
print(f"Working directory: {PROJECT_ROOT}")
print(f"Factor period: {FACTOR_START} to {FACTOR_END}")
print(f"Tech period: {TECH_START} to {TECH_END}")

Working directory: /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine
Factor period: 2004-01-01 to 2026-03-31
Tech period: 2016-01-01 to 2026-03-31


## 1.1 Download Kenneth French Data Library Factors

In [2]:
# Fetch Fama-French 5 Factors + Momentum (monthly, decimal returns)
factor_returns = fetch_french_factors(start='2004')

print(f"Factor returns shape: {factor_returns.shape}")
print(f"Date range: {factor_returns.index.min()} to {factor_returns.index.max()}")
print(f"Columns: {list(factor_returns.columns)}")
print(f"\nSanity checks:")
print(f"  Max |return|: {factor_returns.abs().max().max():.4f} (should be < 0.50 in decimal)")
print(f"  Mkt-RF mean: {factor_returns['mkt_rf'].mean():.6f} (expected ~0.004-0.008)")
print(f"  HML mean: {factor_returns['hml'].mean():.6f}")
print(f"  UMD mean: {factor_returns['umd'].mean():.6f}")
print(f"  RMW mean: {factor_returns['rmw'].mean():.6f}")

# Validate: returns in decimal (not percentage)
# Note: extreme monthly returns up to ~35% can occur during crisis/recovery
assert factor_returns.abs().max().max() < 0.50, \
    f"Factor returns look like percentages! Max = {factor_returns.abs().max().max():.2f}"
logger.info(f"French factors: {factor_returns.shape}, range {factor_returns.index.min()} to {factor_returns.index.max()}")
factor_returns.head()

Factor returns shape: (265, 7)
Date range: 2004-01-31 00:00:00 to 2026-01-31 00:00:00
Columns: ['mkt_rf', 'smb', 'hml', 'rmw', 'cma', 'rf', 'umd']

Sanity checks:
  Max |return|: 0.3434 (should be < 0.50 in decimal)
  Mkt-RF mean: 0.008226 (expected ~0.004-0.008)
  HML mean: -0.000212
  UMD mean: 0.001190
  RMW mean: 0.003038


,mkt_rf,smb,hml,rmw,cma,rf,umd
date,,,,,,,
2004-01-31,0.0215,0.0246,0.0242,-0.0360,0.0333,0.0007,0.0257
2004-02-29,0.0140,-0.0092,0.0095,0.0213,-0.0117,0.0006,-0.0117
2004-03-31,-0.0132,0.0211,0.0023,0.0165,-0.0098,0.0009,0.0019
2004-04-30,-0.0182,-0.0206,-0.0323,0.0346,-0.0283,0.0008,-0.0534
2004-05-31,0.0118,-0.0040,-0.0018,-0.0117,0.0001,0.0006,0.0149


## 1.2 Download FRED Macro Data

In [3]:
# Fetch and transform FRED macro series
# Try fredapi first; fall back to direct FRED CSV download if no API key
import os, io, requests as _requests

FRED_API_KEY = os.environ.get('FRED_API_KEY')

if FRED_API_KEY and FRED_API_KEY != 'your_key_here':
    print("Using fredapi with API key")
    from src.data_loader import fetch_fred_series
    macro_indicators = fetch_fred_series(FRED_SERIES, start='2003-01-01')
else:
    print("No FRED API key found — downloading CSVs directly from FRED website")
    session = _requests.Session()
    session.headers.update({
        'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36',
        'Accept': 'text/csv,text/plain,*/*',
    })
    
    raw = {}
    for code, desc in FRED_SERIES.items():
        try:
            url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={code}&cosd=2003-01-01"
            resp = session.get(url, timeout=30)
            resp.raise_for_status()
            df = pd.read_csv(io.StringIO(resp.text), index_col=0, parse_dates=True, na_values='.')
            raw[code] = pd.to_numeric(df.iloc[:, 0], errors='coerce')
            print(f"  OK {code} ({desc}): {len(raw[code])} obs")
        except Exception as e:
            print(f"  WARN {code} ({desc}): {e}")
        time.sleep(0.3)  # gentle rate limiting
    
    # Resample to monthly and apply transforms (same as data_loader)
    monthly_raw = {code: s.resample('ME').mean() for code, s in raw.items()}
    
    indicators = pd.DataFrame(index=monthly_raw['T10Y2Y'].index)
    indicators.index.name = 'date'
    
    # Direct levels
    indicators['t10y2y'] = monthly_raw['T10Y2Y']
    indicators['baa10y'] = monthly_raw['BAA10Y']
    indicators['vix'] = monthly_raw['VIXCLS']
    indicators['oecd_cli'] = monthly_raw['USALOLITONOSTSAM']
    indicators['unrate'] = monthly_raw['UNRATE']
    
    # Rate of change transforms
    indicators['claims_roc'] = monthly_raw['ICSA'].pct_change(3)
    indicators['oil_roc'] = monthly_raw['DCOILWTICO'].pct_change(3)
    indicators['indpro_yoy'] = monthly_raw['INDPRO'].pct_change(12)
    indicators['unrate_chg'] = monthly_raw['UNRATE'].diff(12)
    
    # Real M2 (deflated by CPI, then 12-month % change)
    real_m2 = monthly_raw['M2SL'] / monthly_raw['CPIAUCSL'] * 100
    indicators['real_m2_yoy'] = real_m2.pct_change(12)
    
    indicators = indicators.ffill(limit=3)
    macro_indicators = indicators

print(f"\nMacro indicators shape: {macro_indicators.shape}")
print(f"Date range: {macro_indicators.index.min()} to {macro_indicators.index.max()}")
print(f"Columns: {list(macro_indicators.columns)}")
check_nan_propagation(macro_indicators, "Macro indicators (before trimming)")

# Display summary stats
macro_indicators.describe().round(4)

No FRED API key found — downloading CSVs directly from FRED website


  OK T10Y2Y (Yield curve slope): 6062 obs


  OK BAA10Y (Credit spread): 6061 obs


  OK VIXCLS (VIX): 6061 obs


  OK ICSA (Initial jobless claims): 1212 obs


...

  OK DCOILWTICO (WTI crude oil): 6058 obs


  OK INDPRO (Industrial production): 278 obs


  OK UNRATE (Unemployment rate): 278 obs



Macro indicators shape: (279, 10)
Date range: 2003-01-31 00:00:00 to 2026-03-31 00:00:00
Columns: ['t10y2y', 'baa10y', 'vix', 'oecd_cli', 'unrate', 'claims_roc', 'oil_roc', 'indpro_yoy', 'unrate_chg', 'real_m2_yoy']


,t10y2y,baa10y,vix,oecd_cli,unrate,claims_roc,oil_roc,indpro_yoy,unrate_chg,real_m2_yoy
count,279.0000,279.0000,279.0000,256.0000,279.0000,276.0000,276.0000,267.0000,267.0000,267.0000
mean,1.0682,2.4189,19.1102,99.8506,5.7312,0.1144,0.0279,0.0059,-0.0745,0.0354
std,0.9659,0.7571,7.9482,1.2840,2.0157,1.4675,0.1987,0.0445,1.8536,0.0543
min,-0.9290,1.4095,10.1255,93.4837,3.4000,-0.7033,-0.7123,-0.1732,-8.7000,-0.0913
25%,0.2398,1.8544,13.9775,99.2744,4.2000,-0.0584,-0.0739,-0.0085,-0.7000,0.0118
50%,1.0141,2.2459,17.2715,99.9542,5.0000,-0.0188,0.0348,0.0175,-0.4000,0.0304
75%,1.8591,2.8257,21.8129,100.6882,6.7000,0.0293,0.1313,0.0288,0.2000,0.0531
max,2.8342,6.0136,62.6689,101.9542,14.8000,20.6392,1.4602,0.1655,11.1000,0.2469


## 1.3 Download S&P 500 Price Data

In [4]:
# Get current S&P 500 constituents
sp500_tickers = fetch_sp500_tickers()
print(f"S&P 500 tickers: {len(sp500_tickers)}")

# Download prices in batches (with rate-limit handling)
sp500_prices = fetch_prices_batch(
    sp500_tickers,
    start='2004-01-01', end='2025-12-31',
    batch_size=50, pause=2.0
)

print(f"\nS&P 500 prices shape: {sp500_prices.shape}")
print(f"Date range: {sp500_prices.index.min()} to {sp500_prices.index.max()}")
print(f"Tickers downloaded: {sp500_prices.shape[1]}")

# Validate: no ticker with >5% exact zeros
zero_pct = (sp500_prices == 0).mean()
bad_tickers = zero_pct[zero_pct > 0.05].index.tolist()
if bad_tickers:
    print(f"WARNING: Tickers with >5% zeros (dropping): {bad_tickers}")
    sp500_prices = sp500_prices.drop(columns=bad_tickers)
else:
    print("All tickers pass zero-check")

logger.info(f"S&P 500 prices: {sp500_prices.shape}")

S&P 500 tickers: 503



S&P 500 prices shape: (5534, 503)
Date range: 2004-01-02 00:00:00 to 2025-12-30 00:00:00
Tickers downloaded: 503
All tickers pass zero-check


In [5]:
# Compute S&P 500 monthly returns
sp500_monthly = sp500_prices.resample('ME').last()
sp500_monthly_returns = sp500_monthly.pct_change().dropna(how='all')

# Reshape to long format for parquet
sp500_daily_long = sp500_prices.stack().reset_index()
sp500_daily_long.columns = ['date', 'ticker', 'adj_close']
sp500_daily_long = sp500_daily_long.set_index('date')

sp500_monthly_long = sp500_monthly_returns.stack().reset_index()
sp500_monthly_long.columns = ['date', 'ticker', 'monthly_return']
sp500_monthly_long = sp500_monthly_long.set_index('date')

print(f"S&P 500 daily (long): {sp500_daily_long.shape}")
print(f"S&P 500 monthly returns (long): {sp500_monthly_long.shape}")

S&P 500 daily (long): (2511783, 2)
S&P 500 monthly returns (long): (119385, 2)


## 1.4 Download Tech Portfolio Data

In [6]:
# Fetch 20 tech tickers + benchmarks with SQ->XYZ merger handling
tech_data = fetch_tech_portfolio(
    TICKERS, BENCHMARKS,
    start='2016-01-01', end='2026-02-28'
)

print(f"Tech portfolio shape: {tech_data.shape}")
print(f"Date range: {tech_data.index.min()} to {tech_data.index.max()}")
print(f"Tickers present: {list(tech_data.columns[:25])}")

# Validate: all 20 tickers present
available_tickers = [t for t in TICKERS if t in tech_data.columns]
missing_tickers = [t for t in TICKERS if t not in tech_data.columns]
print(f"\nAvailable tickers: {len(available_tickers)}/{len(TICKERS)}")
if missing_tickers:
    print(f"Missing tickers: {missing_tickers}")

# Validate short-history tickers have correct IPO dates
for ticker, ipo_date in SHORT_HISTORY_TICKERS.items():
    if ticker in tech_data.columns:
        first_valid = tech_data[ticker].first_valid_index()
        print(f"  {ticker}: first data = {first_valid}, expected IPO ~ {ipo_date}")

logger.info(f"Tech portfolio: {tech_data.shape}, {len(available_tickers)} tickers")

Tech portfolio shape: (2556, 27)
Date range: 2016-01-04 00:00:00 to 2026-02-27 00:00:00
Tickers present: ['AAPL', 'ADBE', 'AMD', 'ANET', 'AVGO', 'CRM', 'CRWD', 'DDOG', 'DX-Y.NYB', 'GOOG', 'META', 'MSFT', 'MU', 'NFLX', 'NOW', 'NVDA', 'PANW', 'PLTR', 'QCOM', 'SOXX', 'TSM', 'XLK', '^GSPC', '^NDX', '^VIX']

Available tickers: 20/20
  CRWD: first data = 2019-06-12 00:00:00, expected IPO ~ 2019-06-12
  DDOG: first data = 2019-09-19 00:00:00, expected IPO ~ 2019-09-19
  PLTR: first data = 2020-09-30 00:00:00, expected IPO ~ 2020-09-30


## 1.5 Date Alignment

In [7]:
# Align factor returns and macro indicators to common month-end dates
# Normalise all to month-end
factor_returns.index = factor_returns.index + pd.offsets.MonthEnd(0)
macro_indicators.index = macro_indicators.index + pd.offsets.MonthEnd(0)

# Remove duplicate dates after normalisation
factor_returns = factor_returns[~factor_returns.index.duplicated(keep='last')]
macro_indicators = macro_indicators[~macro_indicators.index.duplicated(keep='last')]

# Inner join on common dates
common_dates = factor_returns.index.intersection(macro_indicators.index)

# Trim to factor timing window: 2005-01 to 2025-12
common_dates = common_dates[
    (common_dates >= '2005-01-01') & (common_dates <= '2025-12-31')
]

factor_returns_aligned = factor_returns.loc[common_dates]
macro_indicators_aligned = macro_indicators.loc[common_dates]

# Drop any remaining NaN rows from macro (warm-up period)
valid_mask = ~macro_indicators_aligned.isna().any(axis=1)
common_valid = common_dates[valid_mask]
factor_returns_aligned = factor_returns_aligned.loc[common_valid]
macro_indicators_aligned = macro_indicators_aligned.loc[common_valid]

print(f"Aligned factor returns: {factor_returns_aligned.shape}")
print(f"Aligned macro indicators: {macro_indicators_aligned.shape}")
print(f"Common date range: {common_valid.min()} to {common_valid.max()}")
print(f"Number of monthly observations: {len(common_valid)}")

# Verify no duplicate dates
assert factor_returns_aligned.index.is_unique, "Duplicate dates in factor returns!"
assert macro_indicators_aligned.index.is_unique, "Duplicate dates in macro indicators!"
assert factor_returns_aligned.index.equals(macro_indicators_aligned.index), "Date mismatch!"

logger.info(f"Aligned: {len(common_valid)} months, {common_valid.min()} to {common_valid.max()}")

Aligned factor returns: (232, 7)
Aligned macro indicators: (232, 10)
Common date range: 2005-01-31 00:00:00 to 2024-04-30 00:00:00
Number of monthly observations: 232


## 1.6 Compute Base Features & Stationarity Tests

In [8]:
from statsmodels.tsa.stattools import adfuller, kpss
from src.validation import stationarity_table

# Log returns for stationarity testing (factor returns are already returns)
print("=== Stationarity Tests on Factor Returns ===")
stat_results = stationarity_table(factor_returns_aligned[['mkt_rf', 'hml', 'umd', 'rmw', 'cma']])
print(stat_results.to_string(index=False))

# Summary statistics
print("\n=== Factor Return Summary Statistics ===")
summary = pd.DataFrame({
    'mean_monthly': factor_returns_aligned.mean(),
    'std_monthly': factor_returns_aligned.std(),
    'ann_return': factor_returns_aligned.mean() * 12,
    'ann_vol': factor_returns_aligned.std() * np.sqrt(12),
    'sharpe': (factor_returns_aligned.mean() * 12) / (factor_returns_aligned.std() * np.sqrt(12)),
    'skewness': factor_returns_aligned.skew(),
    'kurtosis': factor_returns_aligned.kurtosis(),
    'min': factor_returns_aligned.min(),
    'max': factor_returns_aligned.max(),
})
print(summary.round(4).to_string())

=== Stationarity Tests on Factor Returns ===
series   adf_stat   adf_pvalue  kpss_stat  kpss_pvalue  adf_reject_1pct  kpss_fail_reject_5pct  stationary
mkt_rf -14.914007 1.444272e-27   0.174654          0.1             True                   True        True
   hml -12.751065 8.532498e-24   0.051206          0.1             True                   True        True
   umd  -6.179071 6.521818e-08   0.036104          0.1             True                   True        True
   rmw -10.324491 2.950050e-18   0.118056          0.1             True                   True        True
   cma  -7.336000 1.096169e-10   0.051793          0.1             True                   True        True

=== Factor Return Summary Statistics ===
        mean_monthly  std_monthly  ann_return  ann_vol  sharpe  skewness  kurtosis     min     max
mkt_rf        0.0077       0.0451      0.0930   0.1564  0.5946   -0.5150    1.2304 -0.1720  0.1358
smb          -0.0002       0.0261     -0.0026   0.0904 -0.0288    0.2320 

/Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/src/validation.py:165: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  kpss_res = kpss(series, regression='c', nlags='auto')
/Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/src/validation.py:165: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  kpss_res = kpss(series, regression='c', nlags='auto')
/Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/src/validation.py:165: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  kpss_res = kpss(series, regression='c', nlags='auto')
/Users/laurentnguyen/Documents/

## 1.7 Validation Gates

In [9]:
# === VALIDATION GATES ===
print("=" * 60)
print("PHASE 1 VALIDATION GATES")
print("=" * 60)

gates = {}

# Gate 1: Factor returns sanity
mkt_mean = factor_returns_aligned['mkt_rf'].mean()
gates['Mkt-RF mean reasonable'] = 0.001 <= mkt_mean <= 0.015
gates['RMW mean positive'] = factor_returns_aligned['rmw'].mean() > 0
gates['UMD mean positive'] = factor_returns_aligned['umd'].mean() > 0

# Gate 2: Returns in decimal (not percentage)
max_abs = factor_returns_aligned.abs().max().max()
gates[f'Max |return| < 0.50 (got {max_abs:.3f})'] = max_abs < 0.50

# Gate 3: Macro data complete
gates['Macro no NaN'] = not macro_indicators_aligned.isna().any().any()

# Gate 4: Date indices match
gates['Date indices match'] = factor_returns_aligned.index.equals(macro_indicators_aligned.index)

# Gate 5: Tech portfolio complete
gates[f'Tech tickers: {len(available_tickers)}/20 present'] = len(available_tickers) >= 18

# Gate 6: Stationarity (log returns pass ADF at 1%)
all_stationary = True
for col in ['mkt_rf', 'hml', 'umd', 'rmw']:
    adf_p = adfuller(factor_returns_aligned[col].dropna(), autolag='AIC')[1]
    if adf_p >= 0.01:
        all_stationary = False
gates['Factor returns stationary (ADF 1%)'] = all_stationary

# Gate 7: Sufficient observations
gates[f'Observations >= 200 (got {len(factor_returns_aligned)})'] = len(factor_returns_aligned) >= 200

for gate_name, passed in gates.items():
    status = "PASS" if passed else "FAIL"
    print(f"  [{status}] {gate_name}")

n_pass = sum(gates.values())
n_total = len(gates)
print(f"\nResult: {n_pass}/{n_total} gates passed")
logger.info(f"Validation: {n_pass}/{n_total} gates passed")

PHASE 1 VALIDATION GATES
  [PASS] Mkt-RF mean reasonable
  [PASS] RMW mean positive
  [PASS] UMD mean positive
  [PASS] Max |return| < 0.50 (got 0.343)
  [PASS] Macro no NaN
  [PASS] Date indices match
  [PASS] Tech tickers: 20/20 present
  [PASS] Factor returns stationary (ADF 1%)
  [PASS] Observations >= 200 (got 232)

Result: 9/9 gates passed


## 1.8 Export Parquet Files

In [10]:
# === Export all Phase 1 outputs ===

# 1. Factor returns
factor_returns_aligned.to_parquet(PROCESSED_DIR / 'factor_returns.parquet')
print(f"Exported factor_returns.parquet: {factor_returns_aligned.shape}")

# 2. Macro indicators
macro_indicators_aligned.to_parquet(PROCESSED_DIR / 'macro_indicators.parquet')
print(f"Exported macro_indicators.parquet: {macro_indicators_aligned.shape}")

# 3. S&P 500 daily prices (long format)
sp500_daily_long.to_parquet(PROCESSED_DIR / 'sp500_daily_prices.parquet')
print(f"Exported sp500_daily_prices.parquet: {sp500_daily_long.shape}")

# 4. S&P 500 monthly returns (long format)
sp500_monthly_long.to_parquet(PROCESSED_DIR / 'sp500_monthly_returns.parquet')
print(f"Exported sp500_monthly_returns.parquet: {sp500_monthly_long.shape}")

# 5. Tech portfolio master data (adjusted close for all tickers)
tech_data.to_parquet(PROCESSED_DIR / 'master_data.parquet')
print(f"Exported master_data.parquet: {tech_data.shape}")

# Save checksums
checksums = {}
for fname in ['factor_returns.parquet', 'macro_indicators.parquet',
              'sp500_daily_prices.parquet', 'sp500_monthly_returns.parquet',
              'master_data.parquet']:
    fpath = PROCESSED_DIR / fname
    h = hashlib.sha256(open(fpath, 'rb').read()).hexdigest()
    checksums[fname] = h
with open(RAW_DIR / 'checksums.json', 'w') as f:
    json.dump(checksums, f, indent=2)
print(f"\nChecksums saved to {RAW_DIR / 'checksums.json'}")

logger.info("Phase 1 complete — all outputs exported")
print("\n=== Phase 1 Complete ===")

Exported factor_returns.parquet: (232, 7)
Exported macro_indicators.parquet: (232, 10)


Exported sp500_daily_prices.parquet: (2511783, 2)
Exported sp500_monthly_returns.parquet: (119385, 2)
Exported master_data.parquet: (2556, 27)



Checksums saved to /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/data/raw/checksums.json

=== Phase 1 Complete ===
